# IOAI — 2025 Competition Gas Classification (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    os.system('wget -q -O data/train.csv https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-competition-gas-classification/train.csv')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# JOAI 2025 — Gas Classification (multimodal)

> **Japan Olympiad in AI 2025 — Playground Competition**

Classify the gas (`NoGas` / `Perfume` / `Mixture` / `Smoke`) from **three modalities**: two gas-sensor readings (`MQ8`, `MQ5`), a **thermal image** (`images/<image_path_uuid>`), and a text `Caption` describing the thermal image. Score = **macro-F1**.

The official Kaggle test labels are hidden, so we grade on a **held-out split** (`index % 5 == 0`) of `train.csv`. **Submit** `submission.csv` (`index,Gas`).

⚠️ **The baseline below uses only the two gas sensors (~0.87 F1).** It ignores the image and the caption — **add those modalities** (see the model solution) to push F1 higher.

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
df = pd.read_csv('data/train.csv').reset_index(drop=True)
is_val = df.index % 5 == 0
tr, va = df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)
print('train', len(tr), 'val', len(va), '| classes', sorted(df['Gas'].unique()))

In [ ]:
# BASELINE: gas sensors only
clf = HistGradientBoostingClassifier(random_state=0).fit(tr[['MQ8','MQ5']], tr['Gas'])
pred = clf.predict(va[['MQ8','MQ5']])
print('held-out macro-F1:', round(f1_score(va['Gas'], pred, average='macro'), 4))
pd.DataFrame({'index': range(len(va)), 'Gas': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va), '(sensors only — add image + caption to improve)')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)